# LIMS – Einfache Verwendung

## Einführung

LIMS (LLM Interaction Management System) ist eine Python-Bibliothek zur Erfassung, Verwaltung und persistenten Speicherung von Interaktionen mit Large Language Models (LLMs).

Bei der Verwendung eines LLM entstehen neben der eigentlichen Antwort weitere Informationen, die für einen wissenschaftlichen Forschungsprozess relevant sein können. Dazu gehören beispielsweise die verwendeten Prompts, die Antworten des Modells, der bisherige Gesprächsverlauf, Zeitstempel und weitere Metadaten. LIMS ermöglicht es, diese Informationen strukturiert zu erfassen und gemeinsam zu speichern.

Die Bibliothek wurde insbesondere für Python-basierte Forschungsumgebungen wie Jupyter Notebooks entwickelt. Sie kann dabei verschiedene Komponenten für die Kommunikation mit LLMs sowie für die Speicherung und Suche von Forschungsdaten miteinander verbinden.

### Was kann mit LIMS gemacht werden?

LIMS unterstützt unter anderem folgende Aufgabenbereiche:

- **Kommunikation mit Large Language Models:** Anfragen an ein konfiguriertes LLM senden und Antworten empfangen.
- **Verwaltung von Konversationen:** Mehrere aufeinander aufbauende Interaktionen innerhalb einer gemeinsamen Konversation verwalten.
- **Kontextverwaltung:** Zusätzliche Informationen für die Verarbeitung einer Anfrage bereitstellen und verwalten.
- **Datenverwaltung:** Entstehende Interaktionsdaten und zugehörige Informationen strukturiert erfassen und persistent speichern.
- **Metadatenverwaltung:** Automatisch oder manuell zusätzliche Informationen zu gespeicherten Interaktionen erfassen.
- **Semantische Suche:** Gespeicherte Inhalte anhand ihrer inhaltlichen Ähnlichkeit wiederfinden.
- 
### High-Level API

Für die einfache Verwendung stellt LIMS eine **High-Level API** bereit. Sie ist als skriptbasierte Schnittstelle konzipiert und soll insbesondere einen möglichst einfachen Einstieg in die Verwendung von LIMS ermöglichen. Für die Nutzung der High-Level API sind keine detaillierten Kenntnisse über den internen Aufbau der Anwendung oder deren einzelne Komponenten erforderlich.

Die High-Level API übernimmt dabei einen Großteil der notwendigen Verwaltung. Komponenten wie die Kommunikation mit einem LLM, die Speicherung von Interaktionsdaten oder die Verwendung einer Vektordatenbank werden über eine einheitliche Schnittstelle angesprochen. Dadurch können typische Anwendungsszenarien mit wenigen aufeinanderfolgenden Funktionsaufrufen umgesetzt werden.

Die High-Level API ist dabei eine vereinfachte Zugriffsmöglichkeit und stellt nicht die gesamte Funktionalität von LIMS dar. Die zugrunde liegenden Objekte und Komponenten können bei Bedarf auch direkt verwendet werden. Eine zentrale Komponente ist hierbei der `InteractionManager`, der die Schnittstelle zu den verschiedenen Endpunkten und deren Funktionalitäten bildet. Die direkte Verwendung dieser Komponenten ermöglicht eine detailliertere Kontrolle über den Ablauf und die einzelnen Verarbeitungsschritte, erfordert jedoch ein tieferes Verständnis des internen Aufbaus von LIMS, weshalb in diesem Beispiel davon abgesehen wird.

### Aufbau dieses Beispiels

Dieses Notebook zeigt die grundlegende Verwendung der High-Level API anhand einer vollständigen Beispielkonfiguration und einigen Verwendungsszenarien.

Zunächst werden die benötigten Komponenten ausgewählt und mit ihren jeweiligen externen Diensten verbunden. Anschließend wird eine Konversation gestartet und eine erste Anfrage an das LLM gesendet. Darauf aufbauend werden die weiteren Funktionen der Schnittstelle anhand kleiner Beispiele vorgestellt.

Die verwendeten Dienste und Datenbanken sind dabei austauschbar. LIMS stellt für die verschiedenen Komponenten eine einheitliche Schnittstelle bereit, sodass die grundlegende Verwendung der API unabhängig von der konkreten technischen Implementierung bleibt.

### Installation

Bevor LIMS verwendet werden kann, muss die Bibliothek in der Python-Umgebung installiert werden, in der das Jupyter Notebook ausgeführt wird.

Wenn dieses Dokument in Jupyter Notebook oder JupyterLab ausgeführt wird, kann die Installation direkt in einer neuen Code-Zelle durchgeführt werden. Dazu wird `pip` verwendet. `pip` ist das Standardwerkzeug von Python zur Installation von Python-Paketen.

Führen Sie zunächst die folgende Code-Zelle aus. Diese dient rein zur Installation, muss danach nicht wiederholt werden und kann aus dem Dokument entfernt werden.

Die Installation kann je nach Python-Umgebung einige Zeit in Anspruch nehmen. Eine erfolgreiche Installation wird durch eine entsprechende Meldung von pip angezeigt. Falls die Installation mit einem OSError fehlschlägt, besitzen Sie möglicherweise nicht die erforderlichen Berechtigungen, um Python-Pakete in der verwendeten Python-Umgebung zu installieren. Wenden Sie sich in diesem Fall an die für die Verwaltung Ihrer Python- bzw. Jupyter-Umgebung zuständige Person. Falls Sie die Python- bzw. Jupyter-Umgebung selbst verwalten, stellen Sie sicher, dass Sie über die erforderlichen Berechtigungen zur Installation von Python-Paketen verfügen. Unter Windows kann es gegebenenfalls erforderlich sein, die verwendete Umgebung im Administratormodus beziehungsweise mit erhöhten Berechtigungen zu öffnen.

In [ ]:
%pip install git+https://github.com/VoltexRB/LIMS.git

## Importieren der benötigten Dateien und Klassen

Nach der Installation können die für die Verwendung von LIMS benötigten Module und Klassen importiert werden. Für die folgenden Beispiele wird hauptsächlich die **High-Level API** verwendet. Diese stellt eine vereinfachte, skriptbasierte Schnittstelle zu den Funktionen von LIMS bereit und ermöglicht dadurch einen einfachen Einstieg in die Verwendung der Bibliothek.

Die einzelnen Importe werden für unterschiedliche Bereiche der Konfiguration und Verwendung von LIMS benötigt:

- `api` stellt die Funktionen der High-Level API bereit.
- `LLMEnum`, `PersistentEnum` und `VectorEnum` ermöglichen die Auswahl der verwendeten LLM-, Vektor- und Persistenzkomponenten.
- `ConnectionType` wird verwendet, um bei der Verbindung anzugeben, mit welcher Art von Komponente eine Verbindung hergestellt wird.
- `ContextMode` wird für die Konfiguration der Kontextverwaltung verwendet.

Die folgenden Beispiele bauen auf diesen Importen auf. Weitere Importe werden nur ergänzt, wenn sie für ein bestimmtes Beispiel benötigt werden.

In [ ]:
from llm_interaction_manager.api import lims_interface as api
from llm_interaction_manager.api.interaction_manager_factory import LLMEnum, PersistentEnum, VectorEnum
from llm_interaction_manager.core.interaction_manager import ConnectionType
from llm_interaction_manager.utils.settings import ContextMode
import os

## Initialisieren der Schnittstelle über die High-Level API

Nachdem die benötigten Module importiert wurden, muss die Schnittstelle von LIMS zunächst initialisiert werden. Dabei wird festgelegt, welche Implementierungen für die einzelnen Endpunkte verwendet werden sollen.

Im folgenden Beispiel werden LangChain als LLM-Komponente, ChromaDB als Vektordatenbank und MongoDB als persistente Datenbank ausgewählt.

Die ausgewählten Implementierungen werden durch die Initialisierung für die weitere Verwendung hinterlegt. Intern wird dabei der zentrale `InteractionManager` initialisiert. Dieser dient als zentrale Schnittstelle zwischen der High-Level API und den einzelnen Komponenten von LIMS.
Zusätzlich werden die zu den ausgewählten Implementierungen gehörenden Verbindungsklassen für die jeweiligen Endpunkte vorbereitet. Welche konkreten Verbindungen anschließend benötigt werden, wird im nächsten Abschnitt festgelegt.

Die Initialisierung selbst stellt noch keine Verbindung zu den externen Endpunkten her. Sie legt zunächst fest, welche Implementierungen verwendet werden sollen und bereitet die dafür benötigten Komponenten vor.

In [ ]:
api.initialize(llm=LLMEnum.LANGCHAIN, vector=VectorEnum.CHROMADB, persistent=PersistentEnum.MONGODB)

## Definieren der Verbindungsdaten und Aufbau der Verbindung

Nachdem die gewünschten Endpunktimplementierungen ausgewählt wurden, müssen im nächsten Schritt die benötigten Verbindungsdaten festgelegt werden. Diese werden jeweils als `dict` definiert und anschließend verwendet, um die Verbindung zu den einzelnen Endpunkten herzustellen.

Zunächst wird das für die Kommunikation mit TogetherAI benötigte Nutzungstoken aus den Umgebungsvariablen des Systems geladen.

Anschließend werden drei `dict` erstellt, welche die Verbindungsdaten der zuvor ausgewählten Endpunkte enthalten:

- **LangChain** benötigt die Auswahl des zu verwendenden LLM-Modells sowie das Nutzungstoken für TogetherAI.
- **ChromaDB** benötigt die Angabe des Verbindungstyps. In diesem Beispiel wird `PERSISTENT` verwendet, also eine Datenbgankinstanz lokal auf dem ausführenden System. Abhängig vom gewählten Verbindungstyp können weitere Parameter erforderlich sein; für die persistente Verbindung wird hier der Pfad zum Speicherort der Daten angegeben.
- **MongoDB** benötigt die Verbindungsinformationen des Datenbankservers, bestehend aus **Host** und **Port**, sowie die Auswahl der zu verwendenden Datenbank.

Nachdem die Verbindungsdaten für alle Endpunkte definiert wurden, können die Verbindungen über die High-Level API aufgebaut werden.

Hierfür wird `api.connect()` verwendet. Die Funktion erhält den jeweiligen `ConnectionType` und die zugehörigen Verbindungsdaten. Der `ConnectionType` bestimmt dabei, zu welcher Art von Endpunkt die Verbindung hergestellt werden soll. Das übergebene `dict` enthält die für den jeweiligen Endpunkt benötigten Konfigurations- und Verbindungsdaten.

Nachdem die Endpunkte eingerichtet und verbunden wurden, muss für die Verwendung von LIMS zunächst eine Konversation initialisiert werden. Dadurch wird eine neue Konversation angelegt, in deren Rahmen die folgenden Interaktionen erfasst und verwaltet werden.

Zusätzlich wird für die nachfolgenden Beispiele die Einstellung `wait_for_manual_data` zunächst deaktiviert. Die Einstellung wird auch in späteren Beispielen verändert und wird daher hier vorsorglich zurückgesetzt, falls die Durchführung zuvor an einer entsprechenden Stelle abgebrochen wurde.


In [ ]:
#token aus Umgebungsvariable laden
token = os.getenv("together_ai_token")

#Verbindungsdaten
llm_data = {
    "model": "Prism-ML/Ternary-Bonsai-27B",
    "token": token
}
vector_data = {
    "client_type": "PERSISTENT",
    "persistent_client_db_path": "D:/chroma"
}
persistent_data = {
    "host": "localhost",
    "port": 27017,
    "database": "promptDB"
}

api.connect(ConnectionType.LLM, llm_data)
api.connect(ConnectionType.VECTOR, vector_data)
api.connect(ConnectionType.PERSISTENT, persistent_data)

# Checking Connection
print(
    f"Connected: "
    f"LLM: {api.is_connected(ConnectionType.LLM)} "
    f"VECTOR: {api.is_connected(ConnectionType.VECTOR)} "
    f"PERSISTENT: {api.is_connected(ConnectionType.PERSISTENT)}"
)
api.start_conversation()
api.write_setting("wait_for_manual_data", False)

## Anwendungsmöglichkeiten

LIMS kann für verschiedene Aufgaben im Zusammenhang mit der Nutzung und Dokumentation von LLMs eingesetzt werden. Die folgenden Beispiele zeigen diese Möglichkeiten schrittweise anhand der High-Level API.

### 1. Einfache Anfrage an ein LLM

Die grundlegendste Anwendung von LIMS ist die Kommunikation mit einem LLM.

LIMS übernimmt dabei die Verwaltung und Dokumentation dieser Interaktion. Die Anfrage und die zugehörige Antwort werden im Rahmen der zuvor initialisierten Konversation verarbeitet und können entsprechend der konfigurierten Komponenten gespeichert werden.

Im ersten Beispiel wird daher eine einfache Anfrage an das LLM gesendet durch `send_prompt()` und die zurückgegebene Antwort durch ein `print()` ausgegeben.

In [ ]:
response = api.send_prompt("Einfaches Prompt")
print(response["content"])

## 2: Setzen eines Systemprompts

Das Verhalten des LLM kann durch einen **System-Prompt** beeinflusst werden. Dieser dient dazu, grundlegende Vorgaben für die Verarbeitung von Anfragen festzulegen. Beispielsweise kann dadurch bestimmt werden, welche Rolle das LLM einnehmen oder in welcher Form es Antworten geben soll.

Das Systemprompt befindet sich unter dem Schlüssel `system_prompt` in den Einstellungen. Es wird daher wie alle weiteren Einstellungen und Nutzerpräferenzen mit der Methode `write_setting()` beschrieben. Dieser werden der Name der Einstellung und der zugehörige Wert übergeben.

Beim Senden einer Anfrage wird der hinterlegte System-Prompt intern berücksichtigt und zusammen mit der eigentlichen Anfrage an das LLM übermittelt.

In [ ]:
api.write_setting("system_prompt", "Dies ist ein Deployment-Test. Starte jede Antwort mit 'TEST:'")
system_response = api.send_prompt("Prompt zum testen des Systemprompts")
print(system_response["content"])
api.write_setting("system_prompt", "-1")

## 3: Hinzufügen von Kontextdaten direkt und dynamisch

Zusätzlich zum eigentlichen Prompt können dem LLM weitere Informationen als **Kontextdaten** zur Verfügung gestellt werden. Diese können beispielsweise genutzt werden, um einer Anfrage zusätzliche Informationen bereitzustellen, auf deren Grundlage das LLM seine Antwort erstellen kann.

Zunächst können Kontextdaten direkt festgelegt und für die Kommunikation mit dem LLM bereitgestellt werden. LIMS bietet darüber hinaus die Möglichkeit, Kontextdaten dynamisch zu bestimmen. Dabei können bereits vorhandene Daten aus vorherigen Anfragen und Konversationen ausgewertet werden, um für eine neue Anfrage automatisch passende Kontextinformationen bereitzustellen.

Im folgenden Beispiel werden zunächst Kontextdaten direkt festgelegt und anschließend verwendet. Danach wird der Kontextmodus so angepasst, dass Kontextdaten dynamisch aus bereits vorhandenen Anfragen bestimmt werden können.

Dieser Modus wird explizit durch `set_context_mode()` gesetzt oder implizit durch das hinzufügen neuer Kontextdaten durch `set_context_data()`.

In [ ]:
context_data = {"doc1":"Der Himmel ist grün"}

api.set_context_data(context_data, volatile=False)
context_response = api.send_prompt("Welche Farbe hat der Himmel in den Daten?")
print(context_response["content"])

api.delete_context_data()
api.set_context_mode(ContextMode.DYNAMIC)
#in dieser Antwort werden KEINE Kontextdaten vom Nutzer direkt verwendet, sondern aus vorherigen Ergebnissen dynamisch bezogen
dyn_response = api.send_prompt("Welche Farbe hatte der Himmel in bereits beantworteten Abfragen?")
print(dyn_response["content"])

## 4: Ähnlichkeitssuche in der Vektordatenbank

LIMS ermöglicht die semantische Suche nach bereits gespeicherten Anfragen und Antworten. Die Methode `nearest_search_vector()` sucht anhand einer übergebenen Anfrage nach inhaltlich ähnlichen Einträgen in der Vektordatenbank. Über einen Anzahlparameter kann festgelegt werden, wie viele der ähnlichsten Ergebnisse zurückgegeben werden sollen.

Im folgenden Beispiel wird eine Suchanfrage übergeben und der Anzahlparameter auf `1` gesetzt. Dadurch wird nur das ähnlichste Ergebnis zur Anfrage zurückgegeben.

In [ ]:
print(api.nearest_search_vector("Dokumente mit Aussagen über den Himmel", 1, "lims_embeddings"))

## 5: Hinzufügen von Kommentaren während des Laufs

Während die Nachrichten einer Konversation in der persistenten Datenbank gespeichert werden, besteht zusätzlich die Möglichkeit, diesen nachträglich freie Nutzerkommentare hinzuzufügen. Dadurch können beispielsweise Anmerkungen oder zusätzliche Informationen zu einer bereits gespeicherten Nachricht dokumentiert werden. Ein Kommentar kann entweder nachträglich über einen Befehl zu einer bereits abgeschlossenen Nachricht hinzugefügt werden oder während der Verarbeitung einer Nachricht direkt vom Nutzer eingegeben werden.

Im folgenden Beispiel wird die Einstellung `wait_for_manual_data` aktiviert. Dadurch wird die Verarbeitung der Nachricht angehalten und die Nachricht dem Nutzer zur Eingabe eines Kommentars ausgegeben. Nach der Eingabe wird die Verarbeitung fortgesetzt und die Nachricht zusammen mit dem Nutzerkommentar gespeichert.

In [ ]:
print("Setting was: ", api.read_setting("wait_for_manual_data"))

api.write_setting("wait_for_manual_data", True)
kommentar_response = api.send_prompt("Die Antwort dieses Prompt wird für ein Nutzerkommentar gehalten")

print(kommentar_response["content"])
api.write_setting("wait_for_manual_data", False)

## 6: Hinzufügen von nachträglichen Metadaten zu einer Nachricht

Wenn einer bereits gespeicherten Nachricht zusätzliche Informationen hinzugefügt werden sollen, kann dafür die Methode `add_metadata()` verwendet werden. Die zusätzlichen Metadaten können beispielsweise genutzt werden, um eine Nachricht nachträglich mit weiteren Informationen zu versehen.

Im folgenden Beispiel werden der letzten gespeicherten Nachricht der aktuellen Konversation zusätzliche Metadaten hinzugefügt. Anschließend werden diese mit `get_metadata()` zur Überprüfung der Funktion wieder ausgelesen.

In [ ]:
api.add_metadata(False, {"Zusatzkommentar": "Dies ist ein Zusatzkommentar, der ebenfalls zu den Metadaten hinzugefügt wird"})
print(api.get_metadata(False, None))

## Abschluss

Dieses Dokument hat die grundlegenden Anwendungsmöglichkeiten von LIMS anhand ausgewählter Beispiele vorgestellt. Für eine weiterführende Beschreibung der Funktionen und die vollständige Referenz stehen weitere Dokumentationen zur Verfügung.

Der Quellcode von LIMS sowie die **Benutzerdokumentation** und **Entwicklerdokumentation** sind im [GitHub-Repository](https://github.com/VoltexRB/LIMS) verfügbar. Die Dokumentationen befinden sich dort im Verzeichnis `Documentation`.